In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import librosa
import pywt
import joblib

from tqdm import tqdm

from scipy.signal import butter, filtfilt, medfilt
from scipy.stats import kurtosis, skew

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from imblearn.over_sampling import SMOTE

from lightgbm import LGBMClassifier

In [37]:
# ============================================================
# PARAMETERS
# ============================================================

SR_TARGET = 16000

FRAME_SIZE = 0.025
HOP_SIZE   = 0.010

N_FFT   = 1024
N_MELS  = 64
N_MFCC  = 20

LOWCUT  = 100
HIGHCUT = 2500

PREEMPH = 0.97

CONTEXT_FRAMES = 5

MIN_EVENT_MS = 25
MAX_GAP_MS   = 40

RANDOM_STATE = 42

DATASET_DIR = r"E:\Research Project (Prof. Preeti Rao)\Top files by Crackle count"


In [40]:
# ============================================================
# FILTERS
# ============================================================

def butter_bandpass(lowcut, highcut, fs, order=4):

    nyquist = 0.5 * fs

    low = lowcut / nyquist
    high = highcut / nyquist

    b, a = butter(order, [low, high], btype='band')

    return b, a


def bandpass_filter(y, sr):

    b, a = butter_bandpass(LOWCUT, HIGHCUT, sr)

    return filtfilt(b, a, y)


def pre_emphasis(y, coeff=PREEMPH):

    return np.append(y[0], y[1:] - coeff * y[:-1])


# ============================================================
# PREPROCESS AUDIO
# ============================================================

def preprocess_audio(path):

    y, sr = librosa.load(path, sr=SR_TARGET)

    y = librosa.util.normalize(y)

    y = bandpass_filter(y, sr)

    y = pre_emphasis(y)

    return y, sr


# ============================================================
# WAVELET FEATURES
# ============================================================

def extract_wavelet_features(frame):

    coeffs = pywt.wavedec(frame, 'db4', level=3)

    feats = []

    for c in coeffs:

        feats.extend([
            np.mean(c),
            np.std(c),
            np.max(np.abs(c)),
            kurtosis(c),
            skew(c)
        ])

    return np.array(feats)


# ============================================================
# SPECTRAL ENTROPY
# ============================================================

def spectral_entropy(S):

    psd = S / (np.sum(S, axis=0, keepdims=True) + 1e-10)

    entropy = -np.sum(psd * np.log2(psd + 1e-10), axis=0)

    return entropy


# ============================================================
# FEATURE EXTRACTION
# ============================================================

def extract_features(y, sr):

    frame_length = int(FRAME_SIZE * sr)
    hop_length   = int(HOP_SIZE * sr)

    stft = librosa.stft(
        y,
        n_fft=N_FFT,
        hop_length=hop_length,
        win_length=frame_length
    )

    S = np.abs(stft)

    # --------------------------------------------------------
    # Spectral Features
    # --------------------------------------------------------

    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0]

    bandwidth = librosa.feature.spectral_bandwidth(S=S, sr=sr)[0]

    rolloff = librosa.feature.spectral_rolloff(S=S, sr=sr)[0]

    flatness = librosa.feature.spectral_flatness(S=S)[0]

    entropy = spectral_entropy(S)

    # --------------------------------------------------------
    # Time Features
    # --------------------------------------------------------

    zcr = librosa.feature.zero_crossing_rate(
        y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    rms = librosa.feature.rms(
        y=y,
        frame_length=frame_length,
        hop_length=hop_length
    )[0]

    # --------------------------------------------------------
    # Mel Spectrogram + MFCC
    # --------------------------------------------------------

    mel = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_fft=N_FFT,
        hop_length=hop_length,
        n_mels=N_MELS
    )

    mel_db = librosa.power_to_db(mel)

    mfcc = librosa.feature.mfcc(
        S=mel_db,
        n_mfcc=N_MFCC
    )

    delta = librosa.feature.delta(mfcc)

    delta2 = librosa.feature.delta(mfcc, order=2)

    # --------------------------------------------------------
    # Spectral Flux
    # --------------------------------------------------------

    flux = np.sqrt(np.sum(np.diff(S, axis=1) ** 2, axis=0))

    flux = np.concatenate([[0], flux])

    # --------------------------------------------------------
    # Frequency Ratios
    # --------------------------------------------------------

    freqs = librosa.fft_frequencies(sr=sr, n_fft=N_FFT)

    low_band = np.logical_and(freqs >= 100, freqs <= 500)

    crackle_band = np.logical_and(freqs >= 500, freqs <= 2500)

    total_energy = np.sum(S, axis=0) + 1e-10

    low_ratio = np.sum(S[low_band], axis=0) / total_energy

    crackle_ratio = np.sum(S[crackle_band], axis=0) / total_energy

    # --------------------------------------------------------
    # Framewise Features
    # --------------------------------------------------------

    n_frames = S.shape[1]

    features = []

    for i in range(n_frames):

        start = i * hop_length
        end = start + frame_length

        frame = y[start:end]

        if len(frame) < frame_length:

            frame = np.pad(
                frame,
                (0, frame_length - len(frame))
            )

        # ----------------------------------------------------
        # Time Domain
        # ----------------------------------------------------

        time_feats = [
            np.mean(frame),
            np.std(frame),
            kurtosis(frame),
            skew(frame),
            np.max(np.abs(frame)),
            rms[i],
            zcr[i]
        ]

        # ----------------------------------------------------
        # Spectral
        # ----------------------------------------------------

        spectral_feats = [
            centroid[i],
            bandwidth[i],
            rolloff[i],
            flatness[i],
            entropy[i],
            flux[i],
            low_ratio[i],
            crackle_ratio[i]
        ]

        # ----------------------------------------------------
        # MFCC
        # ----------------------------------------------------

        mfcc_feats = mfcc[:, i]

        delta_feats = delta[:, i]

        delta2_feats = delta2[:, i]

        # ----------------------------------------------------
        # Wavelets
        # ----------------------------------------------------

        wavelet_feats = extract_wavelet_features(frame)

        # ----------------------------------------------------
        # Combine
        # ----------------------------------------------------

        feat_vector = np.concatenate([
            time_feats,
            spectral_feats,
            mfcc_feats,
            delta_feats,
            delta2_feats,
            wavelet_feats
        ])

        features.append(feat_vector)

    features = np.array(features)

    return features, hop_length

In [42]:
# ============================================================
# CONTEXT STACKING
# ============================================================

def stack_context(features, context=CONTEXT_FRAMES):

    padded = np.pad(
        features,
        ((context, context), (0, 0)),
        mode='edge'
    )

    stacked = []

    for i in range(context, len(features) + context):

        window = padded[i-context:i+context+1]

        stacked.append(window.flatten())

    return np.array(stacked)


# ============================================================
# LABEL CREATION
# ============================================================

def create_labels(label_path, n_frames, sr, hop_length):

    labels = np.zeros(n_frames, dtype=np.int32)

    df = pd.read_csv(
        label_path,
        sep='\t',
        header=None,
        names=['start', 'end', 'label']
    )

    for _, row in df.iterrows():

        label = str(row['label']).strip()

        if label not in ['I', 'D']:
            continue

        start_frame = int(row['start'] * sr / hop_length)

        end_frame = int(row['end'] * sr / hop_length)

        labels[start_frame:end_frame+1] = 1

    return labels


# ============================================================
# DATASET BUILDER
# ============================================================

def build_dataset(audio_dir):

    X = []
    y = []
    groups = []

    files = sorted([
        f for f in os.listdir(audio_dir)
        if f.endswith(".wav")
    ])

    for file in tqdm(files):

        audio_path = os.path.join(audio_dir, file)

        label_path = os.path.join(
            audio_dir,
            file.replace(".wav", "_label_audacity.txt")
        )

        if not os.path.exists(label_path):

            continue

        # ----------------------------------------------------
        # Load + preprocess
        # ----------------------------------------------------

        signal, sr = preprocess_audio(audio_path)

        # ----------------------------------------------------
        # Extract features
        # ----------------------------------------------------

        feats, hop_length = extract_features(signal, sr)

        feats = stack_context(feats)

        # ----------------------------------------------------
        # Labels
        # ----------------------------------------------------

        labels = create_labels(
            label_path,
            len(feats),
            sr,
            hop_length
        )

        X.append(feats)

        y.append(labels)

        groups.extend([file] * len(labels))

    X = np.vstack(X)

    y = np.concatenate(y)

    groups = np.array(groups)

    return X, y, groups


# ============================================================
# BUILD DATASET
# ============================================================

print("\nBuilding dataset...\n")

X, y, groups = build_dataset(DATASET_DIR)

print("\nDataset shape:", X.shape)
print("Labels shape:", y.shape)

# ============================================================
# TRAIN / TEST SPLIT
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=RANDOM_STATE
)

train_idx, test_idx = next(
    splitter.split(X, y, groups)
)

X_train = X[train_idx]
y_train = y[train_idx]

X_test = X[test_idx]
y_test = y[test_idx]

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

# ============================================================
# SCALING
# ============================================================

print("\nScaling features...\n")

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_test = scaler.transform(X_test)


Building dataset...



  0%|          | 0/115 [00:11<?, ?it/s]


KeyboardInterrupt: 

In [45]:
# ============================================================
# MODEL
# ============================================================

print("\nTraining LightGBM...\n")

pos_weight = np.sum(y_train == 0) / np.sum(y_train == 1)

model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    max_depth=8,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary',
    scale_pos_weight=pos_weight,
    random_state=RANDOM_STATE
)

model.fit(X_train, y_train)

# ============================================================
# PREDICTION
# ============================================================

print("\nPredicting...\n")

probs = model.predict_proba(X_test)[:, 1]

# ============================================================
# SMOOTHING
# ============================================================

smoothed_probs = medfilt(probs, kernel_size=7)

threshold = 0.45

preds = (smoothed_probs >= threshold).astype(np.int32)

# ============================================================
# POST PROCESSING
# ============================================================

def post_process(preds, sr, hop_length):

    min_frames = int(
        (MIN_EVENT_MS / 1000) * sr / hop_length
    )

    max_gap = int(
        (MAX_GAP_MS / 1000) * sr / hop_length
    )

    cleaned = preds.copy()

    # --------------------------------------------------------
    # Fill small gaps
    # --------------------------------------------------------

    i = 0

    while i < len(cleaned):

        if cleaned[i] == 0:

            start = i

            while i < len(cleaned) and cleaned[i] == 0:
                i += 1

            gap = i - start

            if (
                gap <= max_gap
                and start > 0
                and i < len(cleaned)
                and cleaned[start-1] == 1
                and cleaned[i] == 1
            ):

                cleaned[start:i] = 1

        else:

            i += 1

    # --------------------------------------------------------
    # Remove tiny detections
    # --------------------------------------------------------

    i = 0

    while i < len(cleaned):

        if cleaned[i] == 1:

            start = i

            while i < len(cleaned) and cleaned[i] == 1:
                i += 1

            seg_len = i - start

            if seg_len < min_frames:

                cleaned[start:i] = 0

        else:

            i += 1

    return cleaned


preds = post_process(
    preds,
    SR_TARGET,
    int(HOP_SIZE * SR_TARGET)
)

# ============================================================
# EVALUATION
# ============================================================

print("\n================ EVALUATION ================\n")

print(classification_report(y_test, preds))

print("Confusion Matrix:\n")
print(confusion_matrix(y_test, preds))

auc = roc_auc_score(y_test, smoothed_probs)

acc = accuracy_score(y_test, preds)

precision = precision_score(y_test, preds)

recall = recall_score(y_test, preds)

f1 = f1_score(y_test, preds)

print(f"\nAccuracy  : {acc:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {auc:.4f}")


Training LightGBM...

[LightGBM] [Info] Number of positive: 98139, number of negative: 39953
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 1.994273 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 263705
[LightGBM] [Info] Number of data points in the train set: 138092, number of used features: 1045
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.710678 -> initscore=0.898681
[LightGBM] [Info] Start training from score 0.898681

Predicting...


================ EVALUATION ================

              precision    recall  f1-score   support

           0       0.57      0.73      0.64     10056
           1       0.87      0.77      0.82     24467

    accuracy                           0.76     34523
   macro avg       0.72      0.75      0.73     34523
weighted avg       0.78      0.76      0.77     34523

Confusion Matrix:

[[ 7315  2741]
 [ 5597 18870]]

Accuracy  : 0.7585
Precision : 0.8732
R